In [18]:
from langgraph.graph import StateGraph,START,END
from typing import TypedDict
from langchain_google_genai import ChatGoogleGenerativeAI
from dotenv import load_dotenv
from langgraph.checkpoint.memory import InMemorySaver

Benefits of Persistance
1. Short term memory
2. Fault tolerance
3. HITL
4. Time Travel

In [2]:
load_dotenv()

llm = ChatGoogleGenerativeAI(
  model="gemini-2.5-flash",
  temperature=0.7
)

In [3]:
class JokeState(TypedDict):

  topic: str
  joke: str
  explanation: str

In [4]:
def generate_joke(state: JokeState):

  prompt = f'generate a joke on the topic - {state['topic']}'
  response = llm.invoke(prompt).content

  return {'joke': response}

In [5]:
def generate_explanation(state: JokeState):

  prompt = f'write an explanation for the joke - {state["joke"]}'
  response = llm.invoke(prompt).content

  return {'explanation': response}

In [6]:
graph = StateGraph(JokeState)

graph.add_node('generate_joke', generate_joke)
graph.add_node('generate_explanation', generate_explanation)

graph.add_edge(START, 'generate_joke')
graph.add_edge('generate_joke', 'generate_explanation')
graph.add_edge('generate_explanation', END)

checkpointer = InMemorySaver()

workflow = graph.compile(checkpointer=checkpointer)

In [7]:
# Thread 1
config1 = {"configurable":{"thread_id":"1"}}
workflow.invoke({'topic':'pizza'},config=config1)

Direct use of automatic function calling (AFC) in Models.generate_content is not recommended. Instead, we recommend to use AFC in Chat.send_message. Similarly, direct use of AFC in Models.generate_content_stream is not recommended. Instead, we recommend to use AFC in Chat.send_message_stream.


{'topic': 'pizza',
 'joke': 'Why did the pizza get a job?\n\nBecause it **kneaded dough**!',
 'explanation': 'This joke is a classic example of a **pun**, which plays on words that sound alike but have different meanings.\n\nHere\'s the breakdown:\n\n1.  **"Kneaded dough" (literal meaning):** When you make pizza from scratch, you have to *knead* the *dough*. This is a physical action where you work the mixture of flour, water, and other ingredients to develop its texture. So, a pizza (or someone making one) literally "kneads dough."\n\n2.  **"Needed dough" (slang meaning):** This sounds exactly like "kneaded dough."\n    *   "Needed" is the past tense of "need," meaning to require something.\n    *   "Dough" is a common slang term for **money**.\n\n**The Joke\'s Punchline:**\n\nThe humor comes from the fact that the pizza is given a human motivation (getting a job) for a reason that sounds like a human need ("needed money"), but is phrased using a term that also literally applies to pi

In [8]:
workflow.get_state(config1)

StateSnapshot(values={'topic': 'pizza', 'joke': 'Why did the pizza get a job?\n\nBecause it **kneaded dough**!', 'explanation': 'This joke is a classic example of a **pun**, which plays on words that sound alike but have different meanings.\n\nHere\'s the breakdown:\n\n1.  **"Kneaded dough" (literal meaning):** When you make pizza from scratch, you have to *knead* the *dough*. This is a physical action where you work the mixture of flour, water, and other ingredients to develop its texture. So, a pizza (or someone making one) literally "kneads dough."\n\n2.  **"Needed dough" (slang meaning):** This sounds exactly like "kneaded dough."\n    *   "Needed" is the past tense of "need," meaning to require something.\n    *   "Dough" is a common slang term for **money**.\n\n**The Joke\'s Punchline:**\n\nThe humor comes from the fact that the pizza is given a human motivation (getting a job) for a reason that sounds like a human need ("needed money"), but is phrased using a term that also lite

In [13]:
list(workflow.get_state_history(config1))

[StateSnapshot(values={'topic': 'pizza', 'joke': 'Why did the pizza get a job?\n\nBecause it **kneaded dough**!', 'explanation': 'This joke is a classic example of a **pun**, which plays on words that sound alike but have different meanings.\n\nHere\'s the breakdown:\n\n1.  **"Kneaded dough" (literal meaning):** When you make pizza from scratch, you have to *knead* the *dough*. This is a physical action where you work the mixture of flour, water, and other ingredients to develop its texture. So, a pizza (or someone making one) literally "kneads dough."\n\n2.  **"Needed dough" (slang meaning):** This sounds exactly like "kneaded dough."\n    *   "Needed" is the past tense of "need," meaning to require something.\n    *   "Dough" is a common slang term for **money**.\n\n**The Joke\'s Punchline:**\n\nThe humor comes from the fact that the pizza is given a human motivation (getting a job) for a reason that sounds like a human need ("needed money"), but is phrased using a term that also lit

In [15]:
config2 = {"configurable": {"thread_id":"2"}}
workflow.invoke({'topic':'pasta'}, config=config2)

{'topic': 'pasta',
 'joke': 'Why did the tomato sauce break up with the spaghetti?\n\nBecause it found out the spaghetti was an **impasta**!',
 'explanation': 'This joke is a classic example of a **pun**! Here\'s why it\'s funny:\n\n1.  **The Setup:** The joke sets up a human-like scenario (a breakup) between inanimate food items (tomato sauce and spaghetti). This is already a bit silly and unexpected.\n\n2.  **The Punchline: "Impasta"**\n    *   **Sounds like "Impostor":** The word "impasta" sounds almost exactly like "impostor."\n    *   **What is an "Impostor"?** An impostor is someone who pretends to be someone else, a fake, or someone who is not genuine.\n    *   **What is "Pasta"?** Spaghetti is a type of pasta.\n\n3.  **The Pun Explained:**\n    The joke uses "impasta" to cleverly combine two ideas:\n    *   The spaghetti is literally a type of **pasta**.\n    *   By calling it an "impasta," the joke implies that the spaghetti was an **impostor** – meaning it was being fake, unt

In [16]:
workflow.get_state(config1)

StateSnapshot(values={'topic': 'pizza', 'joke': 'Why did the pizza get a job?\n\nBecause it **kneaded dough**!', 'explanation': 'This joke is a classic example of a **pun**, which plays on words that sound alike but have different meanings.\n\nHere\'s the breakdown:\n\n1.  **"Kneaded dough" (literal meaning):** When you make pizza from scratch, you have to *knead* the *dough*. This is a physical action where you work the mixture of flour, water, and other ingredients to develop its texture. So, a pizza (or someone making one) literally "kneads dough."\n\n2.  **"Needed dough" (slang meaning):** This sounds exactly like "kneaded dough."\n    *   "Needed" is the past tense of "need," meaning to require something.\n    *   "Dough" is a common slang term for **money**.\n\n**The Joke\'s Punchline:**\n\nThe humor comes from the fact that the pizza is given a human motivation (getting a job) for a reason that sounds like a human need ("needed money"), but is phrased using a term that also lite

In [17]:
list(workflow.get_state_history(config1))

[StateSnapshot(values={'topic': 'pizza', 'joke': 'Why did the pizza get a job?\n\nBecause it **kneaded dough**!', 'explanation': 'This joke is a classic example of a **pun**, which plays on words that sound alike but have different meanings.\n\nHere\'s the breakdown:\n\n1.  **"Kneaded dough" (literal meaning):** When you make pizza from scratch, you have to *knead* the *dough*. This is a physical action where you work the mixture of flour, water, and other ingredients to develop its texture. So, a pizza (or someone making one) literally "kneads dough."\n\n2.  **"Needed dough" (slang meaning):** This sounds exactly like "kneaded dough."\n    *   "Needed" is the past tense of "need," meaning to require something.\n    *   "Dough" is a common slang term for **money**.\n\n**The Joke\'s Punchline:**\n\nThe humor comes from the fact that the pizza is given a human motivation (getting a job) for a reason that sounds like a human need ("needed money"), but is phrased using a term that also lit

TIME TRAVEL

 debugging me helpful hota hai 
 wapas se generate joke wale step pe jayenge after executing whole once

In [ ]:
# state mil gaya us particular checkpoint id ka
workflow.get_state({"configurable":{"thread_id":"1","checkpoint_id":"1f1b25a7-bc67-624d-8000-32516228baec"}})

StateSnapshot(values={'topic': 'pizza'}, next=('generate_joke',), config={'configurable': {'thread_id': '1', 'checkpoint_id': '1f1b25a7-bc67-624d-8000-32516228baec'}}, metadata={'source': 'loop', 'step': 0, 'parents': {}}, created_at='2026-09-17T05:41:51.859951+00:00', parent_config={'configurable': {'thread_id': '1', 'checkpoint_ns': '', 'checkpoint_id': '1f1b25a7-bc5c-6c77-bfff-89927645f601'}}, tasks=(PregelTask(id='454c1a41-508f-3f7e-5228-f6a65727b176', name='generate_joke', path=('__pregel_pull', 'generate_joke'), error=None, interrupts=(), state=None, result={'joke': 'Why did the pizza get a job?\n\nBecause it **kneaded dough**!'}),), interrupts=())

In [20]:
# workflow uss state se age chalega 
workflow.invoke(None,{"configurable":{"thread_id":"1","checkpoint_id":"1f1b25a7-bc67-624d-8000-32516228baec"}})

{'topic': 'pizza',
 'joke': "Why did the pizza quit its job?\n\nBecause it wasn't getting enough dough, and it just wasn't rising to the occasion!",
 'explanation': 'This joke is a classic example of **wordplay** and **puns**, using words and phrases that have double meanings.\n\nLet\'s break it down:\n\n1.  **"Why did the pizza quit its job?"**\n    *   This sets up the premise that a pizza is like a person who can have a job and quit it.\n\n2.  **"Because it wasn\'t getting enough dough..."**\n    *   **Literal Pizza Meaning:** Pizza is made from *dough* (the mixture of flour, water, yeast, etc., that forms the base).\n    *   **Figurative/Slang Meaning:** "Dough" is a common slang term for **money**. If someone isn\'t getting enough dough, it means they aren\'t earning enough money at their job.\n    *   **The Pun:** The joke plays on the idea that the "pizza" (as if it were a person) wasn\'t making enough money, using a word that literally applies to its own composition.\n\n3.  **"

In [21]:
# abb do naye state aa jayenge kyuki time travel ki wajah se apan purane state pe gye or baki ka wapas se execute kara 
list(workflow.get_state_history(config1))

[StateSnapshot(values={'topic': 'pizza', 'joke': "Why did the pizza quit its job?\n\nBecause it wasn't getting enough dough, and it just wasn't rising to the occasion!", 'explanation': 'This joke is a classic example of **wordplay** and **puns**, using words and phrases that have double meanings.\n\nLet\'s break it down:\n\n1.  **"Why did the pizza quit its job?"**\n    *   This sets up the premise that a pizza is like a person who can have a job and quit it.\n\n2.  **"Because it wasn\'t getting enough dough..."**\n    *   **Literal Pizza Meaning:** Pizza is made from *dough* (the mixture of flour, water, yeast, etc., that forms the base).\n    *   **Figurative/Slang Meaning:** "Dough" is a common slang term for **money**. If someone isn\'t getting enough dough, it means they aren\'t earning enough money at their job.\n    *   **The Pun:** The joke plays on the idea that the "pizza" (as if it were a person) wasn\'t making enough money, using a word that literally applies to its own com

Updating State

In [23]:
workflow.update_state({"configurable":{"thread_id": "1","checkpoint_id":"1f1b25a7-bc67-624d-8000-32516228baec","checkpoint_ns":""}},{'topic':'samosa'})

{'configurable': {'thread_id': '1',
  'checkpoint_ns': '',
  'checkpoint_id': '1f1b2818-54c1-6b00-8001-05563a1674ef'}}

In [24]:
list(workflow.get_state_history(config1))

[StateSnapshot(values={'topic': 'samosa'}, next=('generate_joke',), config={'configurable': {'thread_id': '1', 'checkpoint_ns': '', 'checkpoint_id': '1f1b2818-54c1-6b00-8001-05563a1674ef'}}, metadata={'source': 'update', 'step': 1, 'parents': {}}, created_at='2026-09-17T10:21:18.207788+00:00', parent_config={'configurable': {'thread_id': '1', 'checkpoint_ns': '', 'checkpoint_id': '1f1b25a7-bc67-624d-8000-32516228baec'}}, tasks=(PregelTask(id='11c4b1c0-202b-34f9-9441-5bc9f628d035', name='generate_joke', path=('__pregel_pull', 'generate_joke'), error=None, interrupts=(), state=None, result=None),), interrupts=()),
 StateSnapshot(values={'topic': 'pizza', 'joke': "Why did the pizza quit its job?\n\nBecause it wasn't getting enough dough, and it just wasn't rising to the occasion!", 'explanation': 'This joke is a classic example of **wordplay** and **puns**, using words and phrases that have double meanings.\n\nLet\'s break it down:\n\n1.  **"Why did the pizza quit its job?"**\n    *   Thi

In [28]:
list(workflow.get_state_history(config2))

[StateSnapshot(values={'topic': 'pasta', 'joke': 'Why did the tomato sauce break up with the spaghetti?\n\nBecause it found out the spaghetti was an **impasta**!', 'explanation': 'This joke is a classic example of a **pun**! Here\'s why it\'s funny:\n\n1.  **The Setup:** The joke sets up a human-like scenario (a breakup) between inanimate food items (tomato sauce and spaghetti). This is already a bit silly and unexpected.\n\n2.  **The Punchline: "Impasta"**\n    *   **Sounds like "Impostor":** The word "impasta" sounds almost exactly like "impostor."\n    *   **What is an "Impostor"?** An impostor is someone who pretends to be someone else, a fake, or someone who is not genuine.\n    *   **What is "Pasta"?** Spaghetti is a type of pasta.\n\n3.  **The Pun Explained:**\n    The joke uses "impasta" to cleverly combine two ideas:\n    *   The spaghetti is literally a type of **pasta**.\n    *   By calling it an "impasta," the joke implies that the spaghetti was an **impostor** – meaning it

In [ ]:
workflow.invoke(None,{"configurable":{"thread_id":"2","chechpoint_id":"1f1b27ed-0841-62e5-8001-e9089bef0138"}})

{'topic': 'samosa',
 'joke': "Why did the samosa get sent to the principal's office?\n\nBecause it was always **getting into hot water**!",
 'explanation': 'This joke is a pun that plays on the phrase "**getting into hot water**."\n\nHere\'s the breakdown:\n\n1.  **Literal Meaning (for a Samosa):** Samosas are traditionally deep-fried in very **hot oil**. So, in a literal sense, a samosa *does* spend time "getting into hot water" (or hot oil, which serves the same purpose for the pun) as part of its preparation.\n\n2.  **Idiomatic Meaning (for a person/student):** The idiom "to get into hot water" means to **get into trouble or a difficult situation**. When a student is sent to the principal\'s office, it\'s usually because they\'ve "gotten into hot water" by misbehaving.\n\nThe humor comes from the clever double meaning. The joke anthropomorphizes the samosa (gives it human qualities like going to school) and then uses its literal cooking process to explain why it\'s "in trouble" in t

In [ ]:
# take care of which checkpoint id you are putting
workflow.invoke(None,{"configurable":{"thread_id":"1","chechpoint_id":"1f1b27ed-69f4-6443-8002-a4740396fdb3"}})

{'topic': 'samosa',
 'joke': "Why did the samosa get sent to the principal's office?\n\nBecause it was always **getting into hot water**!",
 'explanation': 'This joke is a pun that plays on the phrase "**getting into hot water**."\n\nHere\'s the breakdown:\n\n1.  **Literal Meaning (for a Samosa):** Samosas are traditionally deep-fried in very **hot oil**. So, in a literal sense, a samosa *does* spend time "getting into hot water" (or hot oil, which serves the same purpose for the pun) as part of its preparation.\n\n2.  **Idiomatic Meaning (for a person/student):** The idiom "to get into hot water" means to **get into trouble or a difficult situation**. When a student is sent to the principal\'s office, it\'s usually because they\'ve "gotten into hot water" by misbehaving.\n\nThe humor comes from the clever double meaning. The joke anthropomorphizes the samosa (gives it human qualities like going to school) and then uses its literal cooking process to explain why it\'s "in trouble" in t